<a href="https://colab.research.google.com/github/YudiH11/MSSP6070/blob/master/WeeklyModules/Week11/PARTICIPATION_ACTIVITY_Week_11_Plotting_geospatial_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import json
import folium

pop = pd.read_csv("https://raw.githubusercontent.com/YudiH11/MSSP6070/master/WeeklyModules/Week11/CA_Legislative_Districts.csv", dtype={"DISTRICT": str})
pop.head()

In [ ]:
import requests
url = "https://raw.githubusercontent.com/YudiH11/MSSP6070/master/WeeklyModules/Week11/california-congressional-districts.geojson"
response = requests.get(url)
response.raise_for_status()

old_geo = response.json()

len(old_geo["features"])

In [ ]:
m = folium.Map(location=[37.0, -119.5], zoom_start=6, tiles="CartoDB positron")
folium.GeoJson(old_geo, tooltip=folium.GeoJsonTooltip(fields=["Name"])).add_to(m)
m

In [ ]:
import requests

url = ("https://services3.arcgis.com/"
    "uknczv4rpevve42E/arcgis/rest/services/"
    "California_US_Congressional_Districts_Map_2020/"
    "FeatureServer/0/query")

params = {"where": "1=1",
    "outFields": "DISTRICT",
    "outSR": 4326,
    "returnGeometry": "true",
    "f": "geojson"}

response = requests.get(url, params=params)
response.raise_for_status()

current_geo = response.json()
len(current_geo["features"])

In [ ]:
pop_by_district = pop.set_index("DISTRICT")["POPULATION"].to_dict()

for feature in current_geo["features"]:
    district = str(feature["properties"]["DISTRICT"]).zfill(2)

    feature["properties"]["DISTRICT"] = district
    feature["properties"]["POPULATION"] = (pop_by_district.get(district))

In [ ]:
heatmap = folium.Map(location=[37.0, -119.5], zoom_start=6, tiles="CartoDB positron")

folium.Choropleth(
    geo_data=current_geo,
    data=pop,
    columns=["DISTRICT", "POPULATION"],
    key_on="feature.properties.DISTRICT",
    fill_color="YlOrRd",
    bins=[760064.5,
        760065.5,
        760066.5,
        760067.5],
    fill_opacity=0.8,
    line_opacity=0.6,
    legend_name="Population per Congressional District").add_to(heatmap)

heatmap

In [ ]:
folium.GeoJson(current_geo,
    style_function=lambda feature: {
        "fillOpacity": 0,
        "color": "#404040",
        "weight": 0.4},
    tooltip=folium.GeoJsonTooltip(
        fields=["DISTRICT", "POPULATION"],
        aliases=["District:", "Population:"],
        localize=True)).add_to(heatmap)

heatmap